# 🧠 AetherMind: AI Computational Assistant (Llama-3 Edition) 🚀

### Instructions:
1. Go to **Runtime -> Change runtime type -> T4 GPU**
2. Click **Runtime -> Run all**
3. Wait ~5 minutes. A public link will appear at the very bottom.
4. Paste that link into your Claude UI!

---

In [3]:
# ============================================================
# STEP 1: Install Dependencies
# ============================================================
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets gradio

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-e7nx0jx6/unsloth_c822c67f7fc94f9a8e72a11fd53951c3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-e7nx0jx6/unsloth_c822c67f7fc94f9a8e72a11fd53951c3
  Resolved https://github.com/unslothai/unsloth.git to commit 45f060899e56d8a8d218529733075814148f6b19
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 25.2 MB/s eta 0:00:00
  

In [4]:
# ============================================================
# STEP 2: Load Llama-3 8B (4-bit quantized, fits on free T4)
# ============================================================
import os
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = None,
)
print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


Model loaded successfully!


In [5]:
# ============================================================
# STEP 3: Setup LoRA for efficient fine-tuning
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA adapter attached!")

Unsloth 2026.6.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA adapter attached!


In [6]:
# Mount Google Drive for checkpoint saving
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/aethermind_checkpoints', exist_ok=True)
print("Drive mounted. Checkpoints will save to Google Drive.")

ValueError: mount failed

In [ ]:
# ============================================================
# STEP 4: Load & Build the Real AetherMind Dataset
# ============================================================
import json
from datasets import Dataset, load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

system_prompt = "You are AetherMind, an intelligent, helpful, and precise AI assistant specializing in Mathematics, Statistics, Probability, Computer Science, Coding, Cybersecurity, and Web Development, capable of answering all general and conversational questions. Follow rules strictly: 1) Understand intent. 2) Be accurate. 3) Structure responses clearly. 4) Adapt to user style. 5) Answer whatever the user asks conversationally, directly, and professionally."

all_examples = []

# --- Source 1: MetaMathQA (math reasoning, ~1000 examples) ---
print("Loading MetaMathQA...")
metamath = load_dataset("meta-math/MetaMathQA", split="train")
metamath = metamath.shuffle(seed=42).select(range(1000))
for row in metamath:
    all_examples.append({
        "instruction": row["query"],
        "input": "",
        "output": row["response"]
    })
print(f"MetaMathQA loaded: {len(metamath)} examples")

# --- Source 2: CodeAlpaca (coding tasks, ~800 examples) ---
print("Loading CodeAlpaca...")
codealpaca = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
codealpaca = codealpaca.shuffle(seed=42).select(range(800))
for row in codealpaca:
    all_examples.append({
        "instruction": row["instruction"],
        "input": row.get("input", ""),
        "output": row["output"]
    })
print(f"CodeAlpaca loaded: {len(codealpaca)} examples")

# --- Source 3: MATH dataset - statistics/probability subset (~200 examples) ---
print("Loading MATH stats subset...")
try:
    math_ds = load_dataset("lighteval/MATH", "all", split="train", trust_remote_code=True)
    math_stats = math_ds.filter(lambda x: x["type"] in ["counting_and_probability", "statistics"])
    math_stats = math_stats.shuffle(seed=42).select(range(min(200, len(math_stats))))
    for row in math_stats:
        all_examples.append({
            "instruction": row["problem"],
            "input": "",
            "output": row["solution"]
        })
    print(f"MATH stats loaded: {len(math_stats)} examples")
except Exception as e:
    print(f"MATH dataset failed, skipping: {e}")

# --- Source 4: Your 20 custom examples (keep as style anchor) ---
custom_data = [
  {
    "instruction": "Find the derivative of f(x) = x^2 * sin(x) using the product rule.",
    "input": "",
    "output": "Using the Product Rule: let u=x², v=sin(x). Then u'=2x, v'=cos(x). So f'(x) = 2x·sin(x) + x²·cos(x)."
  },
  # ... paste your remaining 19 examples here, or load from file
]
all_examples.extend(custom_data)

print(f"\nTotal examples before dedup: {len(all_examples)}")

# --- Decontaminate: remove GSM8K test overlaps ---
print("Decontaminating against GSM8K test set...")
try:
    gsm8k_test = load_dataset("gsm8k", "main", split="test")
    gsm8k_questions = set(row["question"].strip().lower() for row in gsm8k_test)
    before = len(all_examples)
    all_examples = [ex for ex in all_examples
                    if ex["instruction"].strip().lower() not in gsm8k_questions]
    print(f"Removed {before - len(all_examples)} overlapping GSM8K examples")
except Exception as e:
    print(f"Decontamination skipped: {e}")

# --- Train/Val split ---
import random
random.seed(42)
random.shuffle(all_examples)
split_idx = int(len(all_examples) * 0.9)
train_data = all_examples[:split_idx]
val_data = all_examples[split_idx:]
print(f"\nFinal dataset — Train: {len(train_data)}, Val: {len(val_data)}")

# --- Format to Llama-3 chat template ---
def format_prompts(examples):
    instructions = examples["instruction"]
    inputs_      = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs_, outputs):
        user_msg = instruction
        if input_text and input_text.strip():
            user_msg += f"\n{input_text}"
        messages = [
            {"role": "system",    "content": system_prompt},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)
train_dataset = train_dataset.map(format_prompts, batched=True)
val_dataset   = val_dataset.map(format_prompts, batched=True)

print(f"Datasets formatted and ready!")
print(f"Sample: {train_dataset[0]['text'][:300]}...")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# Calculate max_steps for 2 epochs
effective_batch_size = 1 * 8
max_steps = (len(train_dataset) // effective_batch_size) * 2
print(f"Training for {max_steps} steps (~2 epochs over {len(train_dataset)} examples)")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.03,
        max_steps = max_steps,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        save_steps = 100,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/aethermind_checkpoints",
        dataset_text_field = "text",
        max_seq_length = 2048,
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"Training complete! Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ============================================================
# STEP 6: Quick test!
# ============================================================
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "Hi, who are you?"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("AI says:", response)

In [ ]:
# ============================================================
# STEP 7: Launch the Gradio server (copy the link!)
# ============================================================
import gradio as gr
import traceback

FastLanguageModel.for_inference(model)

def chat(message):
    try:
        messages = [
            {"role": "system", "content": "You are AetherMind, a precise and intelligent computational AI assistant specializing in Mathematics, Statistics, Probability, Computer Science, Coding, Cybersecurity, and Web Development. You run locally on the user's device. You MUST strictly refuse to answer questions unrelated to Mathematics, Statistics, Probability, Computer Science, Coding, Cybersecurity, and Web Development. If asked an out-of-scope question, politely decline and pivot back to your expertise."},
            {"role": "user", "content": message}
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt"
        ).to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return response
    except Exception as e:
        return f"Error: {traceback.format_exc()}"

demo = gr.Interface(fn=chat, inputs="text", outputs="text", title="AetherMind AI")
demo.launch(share=True, debug=True)